<a href="https://colab.research.google.com/github/2024ct05055-subbu/mlops-assignment2/blob/main/MLSO_assignment1_2024CT05055.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center>

Machine Learning System Optimization (MLSO) - Assignment 1  

2024CT05055 - SUBBARAYUDU CHAKALI

</center>

## Problem 1: Throughput

### Problem statement

A web server processes 1 request in 0.2 seconds.
- (1) Calculate the throughput of the server in requests per second.
- (2) If 500 requests arrive, how much total time is required in a sequential system?
- (3) If the same server uses 5 cores and achieves ideal linear scaling, what is the new throughput and the total time required for 500 requests?

### Given

*   **Time per request $t$** = 0.2 s
*   **Number of requests** = 500
*   **Number of cores $N$** = 5 (ideal linear scaling)

### Concept

Throughput is the rate at which work is completed. If one request takes $t$ seconds, a single worker completes $1/t$ requests every second.  
With $N$ independent cores and no overhead (ideal linear scaling), throughput multiplies by $N$.  
Total time for a batch is simply the batch size divided by the throughput.

$$\text{Throughput} = \frac{1}{t}$$
$$\text{Total time} = \frac{\text{Requests}}{\text{Throughput}}$$
$$X_{parallel} = N \times X_{single}$$

### Step-by-step solution

*   **Step 1 — Sequential throughput:** $\text{Throughput} = 1 / 0.2 = 5$ requests per second.
*   **Step 2 — Sequential total time:** $\text{Total time} = 500 \times 0.2 = 500 / 5 = 100$ seconds.
*   **Step 3 — Parallel throughput:** $X_{parallel} = 5 \times 5 = 25$ requests per second.
*   **Step 4 — Parallel total time:** $\text{Total time} = 500 / 25 = 20$ seconds.  
    *   *Note:* The speedup is $100 / 20 = 5\times$, exactly equal to the number of cores, as expected for ideal scaling.

In [1]:
"""Problem 1: Throughput of a web server (sequential vs. 5-core parallel)."""

def throughput(time_per_request):
    """Requests completed per second = 1 / service time."""
    return 1.0 / time_per_request

def total_time(num_requests, throughput_rps):
    """Time to finish a batch = number of requests / throughput."""
    return num_requests / throughput_rps

t_req = 0.2        # seconds per request
requests = 500
cores = 5

# 1) Sequential throughput
x_seq = throughput(t_req)
print(f"1) Sequential throughput      = 1 / {t_req} = {x_seq:.2f} requests/sec")

# 2) Sequential total time
t_seq = total_time(requests, x_seq)
print(f"2) Sequential time for {requests}   = {requests} / {x_seq:.0f} = {t_seq:.2f} sec")

# 3) Parallel with ideal linear scaling
x_par = cores * x_seq
t_par = total_time(requests, x_par)
print(f"3) Parallel throughput ({cores} cores) = {cores} x {x_seq:.0f} = {x_par:.2f} requests/sec")
print(f"   Parallel time for {requests}     = {requests} / {x_par:.0f} = {t_par:.2f} sec")
print(f"   Speedup                     = {t_seq:.0f} / {t_par:.0f} = {t_seq / t_par:.2f}x")


1) Sequential throughput      = 1 / 0.2 = 5.00 requests/sec
2) Sequential time for 500   = 500 / 5 = 100.00 sec
3) Parallel throughput (5 cores) = 5 x 5 = 25.00 requests/sec
   Parallel time for 500     = 500 / 25 = 20.00 sec
   Speedup                     = 100 / 20 = 5.00x


## Problem 2: Little's Law

### Problem statement

A cloud application receives 8 requests per second. The average time a request spends in the system is 0.5 seconds.
- (1) Find the average number of requests in the system.  
- (2) If the arrival rate increases to 12 requests per second, the system capacity remains the same, and the average number of requests in the system becomes 9, find the new average time spent in the system.

### Given

*   **Part 1:** $\lambda = 8$ requests/s, $W = 0.5$ s
*   **Part 2:** $\lambda = 12$ requests/s, $L = 9$ requests

### Concept

Little's Law relates three long-run averages of any stable queueing system:
*   $L$: the average number of items in the system
*   $\lambda$: the average arrival rate (equal to throughput in steady state)
*   $W$: the average time an item spends in the system (latency)

$$L = \lambda \cdot W \implies W = \frac{L}{\lambda}$$

It holds regardless of the arrival distribution or service discipline.

### Step-by-step solution

*   **Step 1 — Find $L$:** $L = \lambda \times W = 8 \times 0.5 = 4$. On average, there are 4 requests in the system.
*   **Step 2 — Find the new $W$:** Rearranging the formula: $W = \frac{L}{\lambda} = \frac{9}{12} = 0.75$ seconds.
*   **Step 3 — Compare:** Latency rose from 0.5 s to 0.75 s, a 50% increase, while arrivals rose by 50% (8 $\rightarrow$ 12). Since capacity did not change, more requests are queued (4 $\rightarrow$ 9) and each waits longer.

In [2]:
"""Problem 2: Little's Law  L = lambda * W."""

def avg_in_system(arrival_rate, avg_time):
    """L = lambda * W"""
    return arrival_rate * avg_time

def avg_time_in_system(L, arrival_rate):
    """W = L / lambda"""
    return L / arrival_rate

# Part 1
lam1, W1 = 8, 0.5
L1 = avg_in_system(lam1, W1)
print(f"1) L = lambda x W = {lam1} x {W1} = {L1:.2f} requests in system")

# Part 2
lam2, L2 = 12, 9
W2 = avg_time_in_system(L2, lam2)
print(f"2) W = L / lambda = {L2} / {lam2} = {W2:.2f} sec in system")
print(f"   Latency increase = ({W2} - {W1}) / {W1} x 100 = {(W2 - W1) / W1 * 100:.0f}%")


1) L = lambda x W = 8 x 0.5 = 4.00 requests in system
2) W = L / lambda = 9 / 12 = 0.75 sec in system
   Latency increase = (0.75 - 0.5) / 0.5 x 100 = 50%


## Problem 3: Amdahl's Law

### Problem statement

A weather simulation takes 100 seconds on a single processor. 70% of the execution time can be parallelized; the remaining 30% must run sequentially. The program is run on 4 processors, with the parallel portion perfectly divided and no overhead.
- (1) Calculate the new execution time.  
- (2) Determine the overall speedup.  
- (3) Explain why the speedup is not 4 even though 4 processors are used.

### Given

*   **Single-processor time $T_1$** = 100 s
*   **Parallel fraction $P$** = 0.7; **Sequential fraction $1 - P$** = 0.3
*   **Processors $N$** = 4

### Concept

Amdahl's Law gives the speedup of a fixed-size workload when only a fraction $P$ can be spread across $N$ processors. The sequential part takes the same time no matter how many processors are added, capping the achievable speedup at $1 / (1 - P)$.

$$S(N) = \frac{1}{(1 - P) + \frac{P}{N}}$$
$$T_N = \frac{T_1}{S(N)}$$

### Step-by-step solution

*   **Step 1 — Sequential term:** $1 - P = 1 - 0.7 = 0.3$.
*   **Step 2 — Parallel term:** $P / N = 0.7 / 4 = 0.175$.
*   **Step 3 — Speedup:** $S(4) = \frac{1}{0.3 + 0.175} = \frac{1}{0.475} \approx 2.105$.
*   **Step 4 — New execution time:** $T_4 = \frac{100}{2.105} \approx 47.5$ s.  
    *   *Cross-check in seconds:* The sequential part is $0.3 \times 100 = 30$ s and stays 30 s. The parallel part is 70 s and becomes $70 / 4 = 17.5$ s. Total = $30 + 17.5 = 47.5$ s, which agrees with Step 4.
*   **Step 5 — Why the speedup is not 4:** Only 70% of the work benefits from the extra processors. The 30-second sequential portion still runs on one processor, so at 4 processors it already accounts for $30 / 47.5 \approx 63\%$ of the run time. Even with infinitely many processors, the time can never drop below 30 s, so the maximum speedup is $1 / 0.3 \approx 3.33\times$. The program output below shows the speedup flattening toward that limit as $N$ grows.

In [3]:
"""Problem 3: Amdahl's Law for a weather simulation."""

def amdahl_speedup(P, N):
    """S(N) = 1 / ((1 - P) + P / N)"""
    return 1.0 / ((1 - P) + P / N)

T1 = 100.0   # single-processor time (s)
P = 0.70     # parallel fraction
N = 4        # processors

seq_part = (1 - P)
par_part = P / N
denom = seq_part + par_part
S = amdahl_speedup(P, N)
T_new = T1 / S

print(f"Sequential term (1-P)   = {seq_part:.3f}")
print(f"Parallel term  P/N      = {P}/{N} = {par_part:.3f}")
print(f"Denominator             = {denom:.3f}")
print(f"Speedup S({N})            = 1 / {denom:.3f} = {S:.4f}")
print(f"New execution time      = {T1:.0f} / {S:.4f} = {T_new:.2f} sec")
print(f"  check: seq {T1*seq_part:.0f}s + par {T1*P:.0f}/{N} = {T1*P/N:.1f}s -> {T1*seq_part + T1*P/N:.1f}s")
print(f"Max possible speedup (N->inf) = 1/(1-P) = {1/(1-P):.2f}x")
print("\nSpeedup vs. number of processors:")
for n in [1, 2, 4, 8, 16, 64, 1024]:
    print(f"  N={n:5d}  S={amdahl_speedup(P, n):.3f}")


Sequential term (1-P)   = 0.300
Parallel term  P/N      = 0.7/4 = 0.175
Denominator             = 0.475
Speedup S(4)            = 1 / 0.475 = 2.1053
New execution time      = 100 / 2.1053 = 47.50 sec
  check: seq 30s + par 70/4 = 17.5s -> 47.5s
Max possible speedup (N->inf) = 1/(1-P) = 3.33x

Speedup vs. number of processors:
  N=    1  S=1.000
  N=    2  S=1.538
  N=    4  S=2.105
  N=    8  S=2.581
  N=   16  S=2.909
  N=   64  S=3.216
  N= 1024  S=3.326


## Problem 4: Processor Efficiency

### Problem statement

A scientific data analysis program runs on a single processor in 150 seconds. 85% of the computation can be parallelized, while 15% must run sequentially. The program is executed on 6 processors.
- (1) Calculate the speedup.  
- (2) Determine the efficiency of the parallel system.  
- (3) Explain whether the processors are being fully utilized.

### Given

*   **Single-processor time $T_1$** = 150 s
*   **Parallel fraction $P$** = 0.85; **Sequential fraction $1 - P$** = 0.15
*   **Processors $N$** = 6

### Concept

Speedup is computed with Amdahl's Law. Efficiency measures how much of the purchased processor capacity is doing useful work: it is speedup divided by the number of processors.  
$E = 1$ (100%) means perfect linear scaling; lower values mean processors spend time idle.

$$S(N) = \frac{1}{(1 - P) + \frac{P}{N}}$$
$$E(N) = \frac{S(N)}{N}$$

### Step-by-step solution

*   **Step 1 — Parallel term:** $P / N = 0.85 / 6 \approx 0.1417$.
*   **Step 2 — Denominator:** $0.15 + 0.1417 = 0.2917$.
*   **Step 3 — Speedup:** $S(6) = \frac{1}{0.2917} \approx 3.43$ (exactly $24 / 7 \approx 3.4286$).
*   **Step 4 — Efficiency:** $E(6) = \frac{3.4286}{6} \approx 0.5714 \approx 57.1\%$. *(Note: Dividing the rounded speedup of 3.43 by 6 yields 57.2%; both are acceptable.)*
*   **Step 5 — Parallel run time (additional):** $T_6 = \frac{150}{3.4286} \approx 43.75$ s.  
    *   *Check:* $0.15 \times 150 = 22.5$ s (sequential) + $\frac{0.85 \times 150}{6} = 21.25$ s (parallel) = $43.75$ s.
*   **Step 6 — Are the processors fully utilized?:** No. An efficiency of about 57% means roughly 43% of the total processor capacity is idle. During the 22.5 s sequential phase, five of the six processors have nothing to do. The 15% sequential fraction caps the speedup at $1 / 0.15 \approx 6.67\times$ no matter how many processors are added, and efficiency keeps falling as $N$ increases.

In [4]:
"""Problem 4: Speedup and processor efficiency (Amdahl's Law)."""

def amdahl_speedup(P, N):
    return 1.0 / ((1 - P) + P / N)

def efficiency(S, N):
    """E(N) = S(N) / N"""
    return S / N

T1 = 150.0   # single-processor time (s)
P = 0.85
N = 6

par_term = P / N
denom = (1 - P) + par_term
S = amdahl_speedup(P, N)
E = efficiency(S, N)

print(f"Parallel term P/N   = {P}/{N} = {par_term:.4f}")
print(f"Denominator         = {1-P:.2f} + {par_term:.4f} = {denom:.4f}")
print(f"Speedup S({N})        = 1 / {denom:.4f} = {S:.4f}")
print(f"Efficiency E({N})     = {S:.4f} / {N} = {E:.4f} = {E*100:.2f}%")
print(f"Parallel run time   = {T1:.0f} / {S:.4f} = {T1/S:.2f} sec")
print(f"Idle/wasted capacity= {(1-E)*100:.2f}%")
print(f"Max speedup (N->inf)= 1/(1-P) = {1/(1-P):.2f}x")
print("Fully utilized?    ", "Yes" if E > 0.95 else "No - efficiency well below 100%")


Parallel term P/N   = 0.85/6 = 0.1417
Denominator         = 0.15 + 0.1417 = 0.2917
Speedup S(6)        = 1 / 0.2917 = 3.4286
Efficiency E(6)     = 3.4286 / 6 = 0.5714 = 57.14%
Parallel run time   = 150 / 3.4286 = 43.75 sec
Idle/wasted capacity= 42.86%
Max speedup (N->inf)= 1/(1-P) = 6.67x
Fully utilized?     No - efficiency well below 100%


## Problem 5: Task-Level Parallelism (TLP)

### Problem statement

An image-processing workload is divided into 4 parallel tasks with execution times of 18 s, 20 s, 22 s, and 19 s on different processors. Communication, synchronization, and thread creation add 5 s of overhead. The program takes 80 s on a single processor.
- (1) Determine the parallel execution time including overhead.  
- (2) Calculate the speedup.

### Given

*   **Serial time $T_{serial}$** = 80 s
*   **Task execution times:** $t_1 = 18$ s, $t_2 = 20$ s, $t_3 = 22$ s, $t_4 = 19$ s
*   **Overhead time $T_{overhead}$** = 5 s

### Concept

When independent tasks run simultaneously, the job is finished only when the slowest task finishes. The longest task is the critical path. Any parallel overhead is added on top.

$$T_{parallel} = \max(t_i) + T_{overhead}$$
$$\text{Speedup} = \frac{T_{serial}}{T_{parallel}}$$

### Step-by-step solution

*   **Step 1 — Critical path:** $\max(18, 20, 22, 19) = 22$ s (Task 3).
*   **Step 2 — Parallel time:** $T_{parallel} = 22 + 5 = 27$ s.
*   **Step 3 — Speedup:** $S = \frac{80}{27} \approx 2.96$.
*   **Step 4 — Effect of overhead (additional):** Without overhead, the speedup would be $80 / 22 \approx 3.64$. The 5 s overhead therefore costs about $0.67\times$ of speedup. Efficiency with 4 processors is $2.96 / 4 \approx 74\%$. Load imbalance also contributes: Tasks 1, 2, and 4 sit idle for 4 s, 2 s, and 3 s respectively while waiting for Task 3.

In [5]:
"""Problem 5: Task-Level Parallelism with overhead."""

T_serial = 80.0
tasks = {"Task 1": 18, "Task 2": 20, "Task 3": 22, "Task 4": 19}
T_overhead = 5.0

# Step 1: critical path = longest task
critical_task = max(tasks, key=tasks.get)
t_max = tasks[critical_task]
print(f"Step 1: max(t_i) = max{tuple(tasks.values())} = {t_max} s ({critical_task})")

# Step 2: parallel time
T_parallel = t_max + T_overhead
print(f"Step 2: T_parallel = {t_max} + {T_overhead:.0f} = {T_parallel:.0f} s")

# Step 3: speedup
S = T_serial / T_parallel
print(f"Step 3: Speedup = {T_serial:.0f} / {T_parallel:.0f} = {S:.4f}")

# Extra insight
S_no_ovh = T_serial / t_max
E = S / len(tasks)
print(f"\nSpeedup without overhead = {T_serial:.0f} / {t_max} = {S_no_ovh:.4f}")
print(f"Efficiency (4 procs)     = {S:.4f} / 4 = {E*100:.2f}%")
print("Idle time per processor while waiting for the critical task:")
for name, t in tasks.items():
    print(f"  {name}: {t_max - t} s idle")


Step 1: max(t_i) = max(18, 20, 22, 19) = 22 s (Task 3)
Step 2: T_parallel = 22 + 5 = 27 s
Step 3: Speedup = 80 / 27 = 2.9630

Speedup without overhead = 80 / 22 = 3.6364
Efficiency (4 procs)     = 2.9630 / 4 = 74.07%
Idle time per processor while waiting for the critical task:
  Task 1: 4 s idle
  Task 2: 2 s idle
  Task 3: 0 s idle
  Task 4: 3 s idle


## Problem 6: Memory Performance of a Computing Node

### Problem statement

A computing node has a cache with 5 ns access time and RAM with 100 ns access time. The cache hit rate is 90%.
- (1) Calculate the effective memory access time (EMAT).  
- (2) If the hit rate drops to 70% due to poor locality, determine the new EMAT.  
- (3) If a program took 1 second at a 90% hit rate, estimate its run time at 70%, assuming computation time is directly proportional to EMAT.

### Given

*   **Cache access time $T_{cache}$** = 5 ns
*   **RAM access time $T_{memory}$** = 100 ns
*   **Original Hit Rate $H$** = 0.9, **Degraded Hit Rate $H$** = 0.7
*   **Original runtime $T_{orig}$** = 1 s (at $H = 0.9$)

### Concept

EMAT is the weighted average of access times: a fraction $H$ of accesses hit the cache and cost $T_{cache}$, and the remaining $(1 - H)$ miss and cost $T_{memory}$. This follows the model used on the slides. If run time is proportional to EMAT, the new run time scales by the ratio of the two EMATs.

$$\text{EMAT} = H \cdot T_{cache} + (1 - H) \cdot T_{memory}$$
$$T_{new} = T_{orig} \times \frac{\text{EMAT}_{new}}{\text{EMAT}_{orig}}$$

### Step-by-step solution

*   **Step 1 — EMAT at 90%:** $\text{EMAT} = 0.9 \times 5 + 0.1 \times 100 = 4.5 + 10 = 14.5$ ns.
*   **Step 2 — EMAT at 70%:** $\text{EMAT} = 0.7 \times 5 + 0.3 \times 100 = 3.5 + 30 = 33.5$ ns.
*   **Step 3 — New run time:** $T_{new} = 1 \times \frac{33.5}{14.5} \approx 2.31$ s.
*   **Step 4 — Interpretation:** A 20-percentage-point drop in hit rate makes the program about $2.31\times$ slower (131% longer). The miss term dominates: at 90% hit rate, misses contribute 10 of the 14.5 ns; at 70% they contribute 30 of the 33.5 ns. Because RAM is $20\times$ slower than cache, every extra miss is expensive.

In [6]:
"""Problem 6: Effective Memory Access Time (EMAT)."""

def emat(hit_rate, t_cache, t_mem):
    """EMAT = H*Tcache + (1-H)*Tmemory"""
    return hit_rate * t_cache + (1 - hit_rate) * t_mem

T_cache, T_mem = 5.0, 100.0     # ns
T_orig = 1.0                    # seconds at 90% hit rate

e90 = emat(0.90, T_cache, T_mem)
e70 = emat(0.70, T_cache, T_mem)
print(f"Step 1: EMAT(90%) = 0.9x{T_cache:.0f} + 0.1x{T_mem:.0f} = {0.9*T_cache:.1f} + {0.1*T_mem:.1f} = {e90:.1f} ns")
print(f"Step 2: EMAT(70%) = 0.7x{T_cache:.0f} + 0.3x{T_mem:.0f} = {0.7*T_cache:.1f} + {0.3*T_mem:.1f} = {e70:.1f} ns")
T_new = T_orig * e70 / e90
print(f"Step 3: T_new = {T_orig} x {e70:.1f} / {e90:.1f} = {T_new:.4f} s")
print(f"        Slowdown = {T_new / T_orig:.2f}x ({(T_new/T_orig - 1)*100:.1f}% longer)")

print("\nHit Rate | EMAT (ns) | Exec Time (s)")
for h in [0.99, 0.95, 0.90, 0.80, 0.70, 0.50]:
    e = emat(h, T_cache, T_mem)
    print(f"  {h*100:5.0f}% | {e:9.2f} | {T_orig * e / e90:8.3f}")


Step 1: EMAT(90%) = 0.9x5 + 0.1x100 = 4.5 + 10.0 = 14.5 ns
Step 2: EMAT(70%) = 0.7x5 + 0.3x100 = 3.5 + 30.0 = 33.5 ns
Step 3: T_new = 1.0 x 33.5 / 14.5 = 2.3103 s
        Slowdown = 2.31x (131.0% longer)

Hit Rate | EMAT (ns) | Exec Time (s)
     99% |      5.95 |    0.410
     95% |      9.75 |    0.672
     90% |     14.50 |    1.000
     80% |     24.00 |    1.655
     70% |     33.50 |    2.310
     50% |     52.50 |    3.621


## Problem 7: Comparing Algorithm Growth Rates, Parallel Reduction, and Communication Cost

### Problem statement

*   **Part A:** Two methods have complexities $O(n\sqrt{n})$ (Method A) and $O(n^2)$ (Method B). Estimate the operation counts for $n = 6000$.
*   **Part B:** Results from multiple processors are combined with a tree-based reduction. For $2^{15}$ elements, find the number of reduction stages and the total reduction time if each stage takes 3.5 µs.
*   **Part C:** Using the communication model $\text{Cost} = t_s + m \times t_w$ with $t_s = 45$ µs, $t_w = 3$ ns/byte, and a 5 KB message, find the total communication time.

---

### Part A — Growth rates

*   **Concept:** Method A: $n\sqrt{n}$ vs. Method B: $n^2$
*   **Step 1:** $\sqrt{6000} \approx 77.46$.
*   **Step 2 (Method A):** $6000 \times 77.46 \approx 464,758 \approx 4.65 \times 10^5$ operations. *(The slide shows 464,760 from rounding $\sqrt{6000}$ to 77.46 first.)*
*   **Step 3 (Method B):** $6000^2 = 36,000,000 = 3.6 \times 10^7$ operations.
*   **Step 4 — Compare:** $\frac{n^2}{n\sqrt{n}} = \sqrt{n} \approx 77.46$, so Method B does about $77\times$ more work at $n = 6000$, and the gap widens as $n$ grows. Method A is the better choice for large inputs.

---

### Part B — Parallel reduction

*   **Concept:** In a tree reduction, each stage combines pairs of values, halving the count. Starting from $n$ values, it takes $\log_2 n$ stages to reach one value.
    $$\text{Stages} = \log_2(n)$$
    $$T_{reduce} = \text{Stages} \times t_{stage}$$
*   **Step 1:** $n = 2^{15} = 32,768$ elements.
*   **Step 2:** $\text{Stages} = \log_2(2^{15}) = 15$.
*   **Step 3:** Total reduction time = $15 \times 3.5$ µs = $52.5$ µs. A sequential sum would need 32,767 additions; the tree needs only 15 parallel steps.

---

### Part C — Communication cost

*   **Concept:** $\text{Cost} = t_s + m \times t_w$. This is the same latency–bandwidth model as $T_{comm} = \alpha + \beta \cdot m$: $t_s$ ($\alpha$) is the fixed start-up latency per message and $t_w$ ($\beta$) is the per-byte transfer time.
*   **Step 1 — Message size:** $m = 5 \times 1024 = 5120$ bytes.
*   **Step 2 — Transfer time:** $5120 \times 3\text{ ns} = 15,360\text{ ns} = 15.36$ µs.
*   **Step 3 — Total:** $\text{Cost} = 45 + 15.36 = 60.36$ µs.
*   **Step 4 — Interpretation:** Start-up latency is about 75% of the total. For small messages, latency dominates, which is why distributed training frameworks bucket many small gradient tensors into larger messages before an All-Reduce.

In [7]:
"""Problem 7: Algorithm growth rates, parallel reduction, communication cost."""
import math

# Part A: compare O(n*sqrt(n)) vs O(n^2) at n = 6000
n = 6000
root_n = math.sqrt(n)
ops_A = n * root_n
ops_B = n ** 2
print("Part A: Growth rates (n = 6000)")
print(f"  sqrt(6000)        = {root_n:.4f}")
print(f"  Method A n*sqrt(n)= 6000 x {root_n:.2f} = {ops_A:,.0f}  (~{ops_A:.2e})")
print(f"  Method B n^2      = 6000^2 = {ops_B:,}  (~{ops_B:.2e})")
print(f"  Ratio B/A         = {ops_B / ops_A:.2f}  (= sqrt(n))")

# Part B: tree-based parallel reduction
elements = 2 ** 15
stages = int(math.log2(elements))
t_stage_us = 3.5
print("\nPart B: Parallel reduction")
print(f"  Elements          = 2^15 = {elements:,}")
print(f"  Stages            = log2({elements}) = {stages}")
print(f"  Reduction time    = {stages} x {t_stage_us} us = {stages * t_stage_us:.1f} us")
print(f"  (Sequential sum would need {elements - 1:,} additions)")

# Part C: communication cost  Cost = ts + m * tw
ts_us = 45.0
tw_ns_per_byte = 3.0
m_bytes = 5 * 1024
transfer_us = m_bytes * tw_ns_per_byte / 1000
total_us = ts_us + transfer_us
print("\nPart C: Communication cost")
print(f"  Message size m    = 5 x 1024 = {m_bytes} bytes")
print(f"  Transfer m x tw   = {m_bytes} x {tw_ns_per_byte:.0f} ns = {m_bytes*tw_ns_per_byte:.0f} ns = {transfer_us:.2f} us")
print(f"  Total cost        = {ts_us:.0f} + {transfer_us:.2f} = {total_us:.2f} us")
print(f"  Startup share     = {ts_us / total_us * 100:.1f}% of total")


Part A: Growth rates (n = 6000)
  sqrt(6000)        = 77.4597
  Method A n*sqrt(n)= 6000 x 77.46 = 464,758  (~4.65e+05)
  Method B n^2      = 6000^2 = 36,000,000  (~3.60e+07)
  Ratio B/A         = 77.46  (= sqrt(n))

Part B: Parallel reduction
  Elements          = 2^15 = 32,768
  Stages            = log2(32768) = 15
  Reduction time    = 15 x 3.5 us = 52.5 us
  (Sequential sum would need 32,767 additions)

Part C: Communication cost
  Message size m    = 5 x 1024 = 5120 bytes
  Transfer m x tw   = 5120 x 3 ns = 15360 ns = 15.36 us
  Total cost        = 45 + 15.36 = 60.36 us
  Startup share     = 74.6% of total
